# B 담당 Colab 실행 노트북 (딥 특징 추출 + 평균점 섞기)

실행 전용 노트북이다. 실험 로직은 `src/`의 `.py`에만 있고, 여기서는 환경 준비 · 스크립트 실행 · 결과 훑어보기만 한다.

- 런타임 유형을 **T4 GPU**로 먼저 바꾼 뒤 시작한다(중간에 바꾸면 `/content`가 비워져 데이터를 다시 복사해야 한다).
- 세션을 다시 열면 **셀 1~6을 다시 실행**하고 셀 7(진행 상태)부터 본다. 끝나지 않은 셀은 그대로 다시 실행하면 이어서 진행한다.
- test 분할은 D8 전까지 쓰지 않는다. 평균점 섞기는 `--split dev`만 실행한다.
- 커밋 전에 출력을 지운다.

**셀 1. 설정 (채울 것)**: 저장소 주소, Drive 바로가기 경로 두 개를 채운다.

In [20]:
REPO_URL = "https://github.com/kimminkyung05/puppy.."
DRIVE_DIR = "/content/drive/MyDrive/puppy..."
DATA_DIR  = "/content/drive/MyDrive/Tsinghua Dogs Dataset"
DATA_ZIPS = []   # 비워 두면 셀 3이 저해상도 이미지·주석 zip을 자동으로 찾음
LOCAL_DATA = "/content/data"
REPO = "/content/dog-mix-cv"

**셀 2. Drive 연결, 폴더 확인**: 두 폴더가 보이는지 확인하고 데이터 폴더의 파일 목록을 출력한다.

In [21]:
import os, glob
from google.colab import drive
drive.mount('/content/drive')
for d in [DRIVE_DIR, DATA_DIR]:
    assert os.path.isdir(d), (f"폴더가 없음: {d}\n"
        "→ 링크를 열고 '바로가기 추가 → 내 드라이브'를 했는지, 아래 목록에서 이름을 확인")
print("내 드라이브:", os.listdir("/content/drive/MyDrive"))
print("\n데이터 폴더 내용:")
for p in sorted(glob.glob(f"{DATA_DIR}/**/*", recursive=True))[:100]:
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1e6:10.1f} MB  {os.path.relpath(p, DATA_DIR)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
내 드라이브: ['Colab Notebooks', 'GoodNotes', '2026-1', '25', 'FinGuard', '이상치탐지.gsheet', 'week2-so101실행영상', 'eng', '증명서', 'ppt', 'paper', 'Tsinghua Dogs Dataset', 'Lecture 15-16_Offline_RL.pptx', 'puppy...']

데이터 폴더 내용:
       0.2 MB  TrainValSplit.zip
       2.8 MB  TrainValSplit/TrainAndValList/train.lst
       0.2 MB  TrainValSplit/TrainAndValList/validation.lst
      37.3 MB  low-annotations.zip
       0.0 MB  low-annotations/Low-Annotations/1043-n000001-Shiba_Dog/n100001.jpeg.xml
       0.0 MB  low-annotations/Low-Annotations/1043-n000001-Shiba_Dog/n100002.jpeg.xml
       0.0 MB  low-annotations/Low-Annotations/1043-n000001-Shiba_Dog/n100003.jpeg.xml
       0.0 MB  low-annotations/Low-Annotations/1043-n000001-Shiba_Dog/n100004.jpg.xml
       0.0 MB  low-annotations/Low-Annotations/1043-n000001-Shiba_Dog/n100005.jpg.xml
       0.0 MB  low-

**셀 3. 받을 zip 고르기**: 저해상도 이미지 · 주석 · 분할 zip만 고른다(고해상도 제외).

In [22]:
if not DATA_ZIPS:
    zips = glob.glob(f"{DATA_DIR}/**/*.zip", recursive=True)
    DATA_ZIPS = sorted(
        z for z in zips
        if "high" not in os.path.basename(z).lower()
        and any(k in os.path.basename(z).lower() for k in ["low", "annotation", "split", "trainval"])
    )
print("복사할 zip:", *DATA_ZIPS, sep="\n  ")
assert DATA_ZIPS, "자동으로 못 찾음 → 셀 2 출력을 보고 셀 1의 DATA_ZIPS에 경로를 직접 적기"

복사할 zip:
  /content/drive/MyDrive/Tsinghua Dogs Dataset/TrainValSplit.zip
  /content/drive/MyDrive/Tsinghua Dogs Dataset/low-annotations.zip
  /content/drive/MyDrive/Tsinghua Dogs Dataset/low-resolution.zip


**셀 4. `/content`로 복사 · 압축 해제**: 이번 세션에 이미 푼 zip은 건너뛴다. 마지막 줄에 풀린 폴더 이름이 나온다.

In [23]:
import shutil, subprocess
os.makedirs(LOCAL_DATA, exist_ok=True)
for z in DATA_ZIPS:
    name = os.path.basename(z)
    done = f"{LOCAL_DATA}/.done_{name}"
    if os.path.exists(done):
        print("이미 풀림:", name); continue
    try:
        print("복사:", name); shutil.copy(z, f"/content/{name}")
    except OSError as e:
        raise RuntimeError(f"{name} 복사 실패: {e}\n"
            "→ 공개 폴더 다운로드 한도일 수 있음. 그 zip을 '사본 만들기'로 내 드라이브에 복사한 뒤 "
            "셀 1의 DATA_ZIPS에 그 경로를 적기") from e
    print("압축 해제:", name)
    subprocess.run(["unzip", "-q", "-o", f"/content/{name}", "-d", LOCAL_DATA], check=True)
    os.remove(f"/content/{name}"); open(done, "w").close()
print(os.listdir(LOCAL_DATA))

이미 풀림: TrainValSplit.zip
이미 풀림: low-annotations.zip
이미 풀림: low-resolution.zip
['.done_TrainValSplit.zip', 'Low-Annotations', 'TrainValSplit', '.done_low-annotations.zip', 'low-resolution', '.done_low-resolution.zip']


**셀 5. 코드 받기 + 경로 자동 설정**: 저장소를 clone/pull하고, 풀린 폴더를 찾아 `/content/paths_colab.yaml`을 만들어 `DOGMIX_PATHS`로 지정한다(`configs/paths.yaml`은 고치지 않는다).

In [24]:
import re, yaml
CLASS_RE = re.compile(r"^\d+-n\d+-")
def find_class_root(base, want_xml):
    for dirpath, dirnames, _ in os.walk(base):
        cls = [d for d in dirnames if CLASS_RE.match(d)]
        if len(cls) >= 100:
            sample = os.listdir(os.path.join(dirpath, cls[0]))[:50]
            if any(f.endswith(".xml") for f in sample) == want_xml:
                return dirpath
    return None
def find_file(base, name):
    for dirpath, _, files in os.walk(base):
        if name in files:
            return os.path.join(dirpath, name)
    return None
img_root = find_class_root(LOCAL_DATA, want_xml=False)
ann_root = find_class_root(LOCAL_DATA, want_xml=True)
train_lst, val_lst = find_file(LOCAL_DATA, "train.lst"), find_file(LOCAL_DATA, "validation.lst")
if not (train_lst and val_lst):
    !git clone -q https://github.com/dejungle/Tsinghua-Dogs /content/tsinghua-split
    train_lst = "/content/tsinghua-split/split/train.lst"
    val_lst   = "/content/tsinghua-split/split/validation.lst"
if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
cfg = yaml.safe_load(open("configs/paths.yaml", encoding="utf-8"))
cfg["tsinghua"].update(image_root=img_root, annotation_root=ann_root, train_list=train_lst, val_list=val_lst)
cfg["evalset"]["index_csv"] = f"{DRIVE_DIR}/evalset/eval_index.csv"
cfg["outputs"].update(features_root=f"{DRIVE_DIR}/features", preds_root=f"{DRIVE_DIR}/preds", logs_root=f"{DRIVE_DIR}/logs")
with open("/content/paths_colab.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True)
os.environ["DOGMIX_PATHS"] = "/content/paths_colab.yaml"
print(open("/content/paths_colab.yaml", encoding="utf-8").read())
missing = [k for k, v in {"이미지": img_root, "주석": ann_root, "train.lst": train_lst, "validation.lst": val_lst}.items()
           if not v or not os.path.exists(v)]
assert not missing, f"못 찾음: {missing} → 셀 4의 폴더 목록을 보고 paths_colab.yaml을 직접 고치기"

/content/dog-mix-cv
classes:
  exclude:
  - chinese_rural_dog
  - dhole
  - African_hunting_dog
  exclude_dingo: false
evalset:
  index_csv: /content/drive/MyDrive/puppy.../evalset/eval_index.csv
outputs:
  features_root: /content/drive/MyDrive/puppy.../features
  logs_root: /content/drive/MyDrive/puppy.../logs
  preds_root: /content/drive/MyDrive/puppy.../preds
tsinghua:
  annotation_root: /content/data/Low-Annotations
  image_root: /content/data/low-resolution
  train_list: /content/data/TrainValSplit/TrainAndValList/train.lst
  val_list: /content/data/TrainValSplit/TrainAndValList/validation.lst



**셀 6. GPU 확인, 패키지 설치, Hugging Face 로그인**: DINOv3 접근 승인을 받은 계정의 토큰으로 로그인한다.

In [ ]:
!nvidia-smi
!pip install -q -r requirements.txt
from huggingface_hub import login; login()

**셀 6-1. 데이터 · 모델 점검**: 경로, `bodybndbox` 태그, 몸통 박스를 못 찾은 비율, 클래스 수(127), 모델 ID, 출력 차원, DINOv3 `num_register_tokens`를 출력한다. '확인 필요'가 있으면 추출 전에 해결한다.

In [26]:
!python -m src.common.check_data
!python -m src.features.check_models

== 1) 경로가 실제로 있는지 ==
  설정 파일: /content/paths_colab.yaml
  [있음] tsinghua.image_root: /content/data/low-resolution
  [있음] tsinghua.annotation_root: /content/data/Low-Annotations
  [있음] tsinghua.train_list: /content/data/TrainValSplit/TrainAndValList/train.lst
  [있음] tsinghua.val_list: /content/data/TrainValSplit/TrainAndValList/validation.lst
  [없음] evalset.index_csv: /content/drive/MyDrive/puppy.../evalset/eval_index.csv
  [없음(실행할 때 만들어짐)] outputs.features_root: /content/drive/MyDrive/puppy.../features
  [없음(실행할 때 만들어짐)] outputs.logs_root: /content/drive/MyDrive/puppy.../logs
  [없음(실행할 때 만들어짐)] outputs.preds_root: /content/drive/MyDrive/puppy.../preds

== 2) 분할 파일과 학습 클래스 수 ==
  train 분할 파일: 65228줄 → 제외 후 61592장
  val 분할 파일:   5200줄 → 제외 후 5080장
  학습 클래스: 제외 전 130개 → 제외 후 127개 (기대값 127): OK
  제외 클래스 'chinese_rural_dog': 목록에 있음 → 제외됨
  제외 클래스 'dhole': 목록에 있음 → 제외됨
  제외 클래스 'African_hunting_dog': 목록에 있음 → 제외됨
  dingo: 목록에 있음, exclude_dingo=False

== 3) 주석 xml의 bodybndbox ==
  점검 2000장 (학습

**셀 7. 진행 상태 확인**: 세션을 다시 열면 셀 1~6 실행 후 여기부터 보고, '완료'가 아닌 것을 이어서 실행한다.

In [27]:
!python -m src.common.status --logs 5

== 특징: /content/drive/MyDrive/puppy.../features (샤드 5000장) ==
  (폴더 없음: 아직 추출한 적 없음)

== 예측: /content/drive/MyDrive/puppy.../preds ==
  (폴더 없음: 아직 실행한 적 없음)

== 로그: /content/drive/MyDrive/puppy.../logs/runs.jsonl ==
  (로그 없음)


**셀 8. 첫 실행 확인 (1,000장 시간 재기)**: 샤드가 '완료'이고 `tmp_files`가 0인지(Drive에서 임시 파일 이름 바꾸기가 되는지) 보고, 장/초로 전체 시간을 추정한다.

In [28]:
!python -m src.features.extract_deep --backbone dinov3 --subset train --limit 1000 --tag timing
!python -m src.common.status --logs 1 --eta

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 211/211 [00:00<00:00, 1611.75it/s]
dinov3/train_limit1000 shard 1/1: 100% 8/8 [00:07<00:00,  1.04it/s]
[ok] 1000장, 7.7초, 129.7장/초 → /content/drive/MyDrive/puppy.../features/dinov3/train_limit1000
== 특징: /content/drive/MyDrive/puppy.../features (샤드 5000장) ==
  backbone   subset                  images   shards tmp_files  상태
  dinov3     train_limit1000           1000      1/1         0  완료

== 예측: /content/drive/MyDrive/puppy.../preds ==
  (폴더 없음: 아직 실행한 적 없음)

== 로그: /content/drive/MyDrive/puppy.../logs/runs.jsonl ==
  실행 1건: ok 1, 실패 0
  2026-11-21 14:54:59 | B | FEAT-dinov3-train_limit1000 | ok | images=1000, img_per_sec=129.7, seconds=7.7, tag=timing

== 전체 추출 시간 추정 ==
  기준: FEAT-dinov3-train_limit1000 (2026-11-21 14:54:59), 1000장 / 7.7초 = 129.9장/초, cuda, batch_size=128
  전체 66672장 = 학습 61592 + val 5080 + 평가 0 (eval_index.csv 없음)
  추출기 하나에 약 9분, 3개면 약 26분
  (측정한 추출기 기준. 다른 추출기는 속도가 다를 수 있고, 1,000장

**셀 9. 특징 추출**: dinov3 → dinov2 → clip 순서로 train · val · eval. 끊기면 같은 셀을 다시 실행한다(끝난 샤드는 건너뜀).

In [29]:
for b in ["dinov3", "dinov2", "clip"]:
    for s in ["train", "val", "eval"]:
        !python -m src.features.extract_deep --backbone {b} --subset {s}

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 211/211 [00:00<00:00, 1149.51it/s]
dinov3/train shard 1/13: 100% 40/40 [00:31<00:00,  1.27it/s]
dinov3/train shard 2/13: 100% 40/40 [00:28<00:00,  1.39it/s]
dinov3/train shard 3/13: 100% 40/40 [00:31<00:00,  1.29it/s]
dinov3/train shard 4/13: 100% 40/40 [00:28<00:00,  1.42it/s]
dinov3/train shard 5/13: 100% 40/40 [00:31<00:00,  1.28it/s]
dinov3/train shard 6/13: 100% 40/40 [00:32<00:00,  1.23it/s]
dinov3/train shard 7/13: 100% 40/40 [00:31<00:00,  1.26it/s]
dinov3/train shard 8/13: 100% 40/40 [00:32<00:00,  1.24it/s]
dinov3/train shard 9/13: 100% 40/40 [00:30<00:00,  1.30it/s]
dinov3/train shard 10/13: 100% 40/40 [00:32<00:00,  1.23it/s]
dinov3/train shard 11/13: 100% 40/40 [00:33<00:00,  1.19it/s]
dinov3/train shard 12/13: 100% 40/40 [00:29<00:00,  1.35it/s]
dinov3/train shard 13/13: 100% 13/13 [00:10<00:00,  1.27it/s]
[ok] 61592장, 385.6초, 159.7장/초 → /content/drive/MyDrive/puppy.../features/dinov3/t

**셀 10. 평균점 섞기 (dev)**: 추출기 4종 × L2 없음/있음. 이미 있는 예측은 건너뛴다. `sift`는 C의 특징이 올라온 뒤에 돌아간다.

In [30]:
for b in ["dinov3", "dinov2", "clip", "sift"]:
    !python -m src.rq1.prototype_fcls --backbone {b} --repeats 1 2 3 --split dev --skip-existing
    !python -m src.rq1.prototype_fcls --backbone {b} --repeats 1 2 3 --split dev --skip-existing --l2

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/dog-mix-cv/src/rq1/prototype_fcls.py", line 155, in <module>
    main()
    ~~~~^^
  File "/content/dog-mix-cv/src/rq1/prototype_fcls.py", line 144, in main
    info = run(args.backbone, r, args.split, args.pool, args.l2, args.frac, args.eval_subset, paths)
  File "/content/dog-mix-cv/src/rq1/prototype_fcls.py", line 85, in run
    ids_ev, Xev, iev = load_features(feat_root, backbone, eval_subset, pool)
                       ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/dog-mix-cv/src/common/features_io.py", line 64, in load_features
    raise FileNotFoundError(f"특징 파일 없음: {d}")
FileNotFoundError: 특징 파일 없음: /content/drive/MyDrive/puppy.../features/dinov3/eval
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/

**셀 11. 마무리**: 최종 상태를 보고 Drive에 쓰기를 끝낸 뒤 연결을 끊는다.

In [31]:
!python -m src.common.status
from google.colab import drive; drive.flush_and_unmount()

== 특징: /content/drive/MyDrive/puppy.../features (샤드 5000장) ==
  backbone   subset                  images   shards tmp_files  상태
  clip       train                    61592    13/13         0  완료
  clip       val                       5080      2/2         0  완료
  dinov2     train                    61592    13/13         0  완료
  dinov2     val                       5080      2/2         0  완료
  dinov3     train                    61592    13/13         0  완료
  dinov3     train_limit1000           1000      1/1         0  완료
  dinov3     val                       5080      2/2         0  완료

== 예측: /content/drive/MyDrive/puppy.../preds ==
  (폴더 없음: 아직 실행한 적 없음)

== 로그: /content/drive/MyDrive/puppy.../logs/runs.jsonl ==
  실행 15건: ok 7, 실패 8


In [32]:
!python -m src.common.status --logs 10
from google.colab import drive; drive.flush_and_unmount()

== 특징: /content/drive/MyDrive/puppy.../features (샤드 5000장) ==
  (폴더 없음: 아직 추출한 적 없음)

== 예측: /content/drive/MyDrive/puppy.../preds ==
  (폴더 없음: 아직 실행한 적 없음)

== 로그: /content/drive/MyDrive/puppy.../logs/runs.jsonl ==
  (로그 없음)
Drive not mounted, so nothing to flush and unmount.
